# Sessione 4 · Il tuo primo modello

**Modulo 2 · Python per l'IA** · 2 sessioni · prime righe di codice · strumenti chiave

> Oggi mettiamo insieme tutto quello visto finora: prendiamo dati reali e **addestriamo un modello** che impara da solo. Vedrai che con scikit-learn le idee della Sessione 2 (addestramento, validazione, metriche) stanno in poche righe.

**Al termine della sessione saprai:**
- Addestrare un modello di **classificazione** e uno di **regressione**
- Applicare nella pratica la separazione addestramento / test
- Valutare i risultati con **metriche** e leggere una matrice di confusione
- Visualizzare dati e risultati con **matplotlib**

*Risultato della sessione: addestri un modello, lo valuti e ne leggi i risultati.*

## 1. scikit-learn: la cassetta degli attrezzi del ML

**scikit-learn** (si importa come `sklearn`) è la libreria di riferimento per il Machine Learning classico. Il bello: **tutti i suoi modelli si usano allo stesso modo**, con lo stesso schema in tre passi. Impara questo schema e saprai usare decine di modelli:

1. **Creare** il modello → `modelo = ModeloQueSea()`
2. **Addestrare** con i dati → `modelo.fit(X_entrenamiento, y_entrenamiento)`
3. **Predire** su dati nuovi → `modelo.predict(X_prueba)`

Ricorda il vocabolario della Sessione 2: **`X`** sono le caratteristiche (gli input) e **`y`** è l'etichetta (ciò che vogliamo predire).

## 2. Classificazione: di che specie è questo fiore?

**Classificare** = assegnare una categoria. Torniamo al dataset *iris*: a partire dalle misure di un fiore, il modello predirà la sua **specie** (setosa, versicolor o virginica).

Per prima cosa carichiamo e prepariamo i dati.

In [ ]:
import pandas as pd
from sklearn.datasets import load_iris

iris = load_iris(as_frame=True)
X = iris.data          # le 4 misure (caratteristiche)
y = iris.target        # la specie codificata come 0, 1, 2 (etichetta)

print("Caratteristiche (X):", list(X.columns))
print("Specie possibili (y):", list(iris.target_names))
X.head()

### Passo chiave: separare addestramento e test

Come visto nella Sessione 2, **non si valuta con i dati di studio**. `train_test_split` divide i dati automaticamente. Riserviamo il 20% per l'esame finale.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,        # 20% per il test, 80% per l'addestramento
    random_state=42,      # fissa la casualità per ottenere sempre la stessa suddivisione
    stratify=y,           # mantiene la proporzione di ogni specie in entrambi i gruppi
)
print("Esempi per addestrare:", len(X_train))
print("Esempi per testare  :", len(X_test))

### Addestrare il modello (i tre passi)

Usiamo un **albero decisionale**: un modello che impara domande del tipo "il petalo misura più di 2,5 cm?" ed è facile da interpretare.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

modelo = DecisionTreeClassifier(max_depth=3, random_state=42)  # 1. creare
modelo.fit(X_train, y_train)                                   # 2. addestrare
predicciones = modelo.predict(X_test)                          # 3. predire

print("Predizioni del modello:", predicciones[:10])
print("Valori reali          :", y_test.values[:10])

## 3. Valutare con le metriche

Il modello indovina? La metrica più semplice per la classificazione è l'**accuratezza** (*accuracy*): la percentuale di risposte corrette.

In [ ]:
from sklearn.metrics import accuracy_score

exactitud = accuracy_score(y_test, predicciones)
print(f"Accuratezza: {exactitud:.1%}")    # mostrata come percentuale

### La matrice di confusione

L'accuratezza riassume tutto in un numero, ma nasconde *dove* sbaglia il modello. La **matrice di confusione** mostra, per ogni specie reale, cosa ha predetto il modello. La diagonale sono le risposte corrette; tutto ciò che sta fuori dalla diagonale sono errori.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_estimator(
    modelo, X_test, y_test, display_labels=iris.target_names, cmap="Blues"
)
plt.title("Matrice di confusione")
plt.show()

### 🟢 Esercizio 1

Ripeti l'addestramento, ma cambia il modello con un **K-Vicini** (`KNeighborsClassifier`), un altro classificatore classico. Misura la sua accuratezza e confrontala con quella dell'albero.

```python
from sklearn.neighbors import KNeighborsClassifier
```
*Suggerimento: lo schema crea → fit → predict è identico.*

In [ ]:
# TODO: addestra un KNeighborsClassifier e misura la sua accuratezza


**Soluzione:**

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
pred_knn = knn.predict(X_test)
print(f"Accuratezza K-Vicini: {accuracy_score(y_test, pred_knn):.1%}")

## 4. Regressione: predire un numero

**Regressione** = predire un valore numerico continuo (un prezzo, una temperatura, un'evoluzione), non una categoria. Usiamo un dataset reale sull'evoluzione del diabete: a partire da vari dati clinici si predice un **indicatore numerico di progressione** della malattia.

In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split

datos = load_diabetes(as_frame=True)
Xr = datos.data
yr = datos.target      # un numero continuo (non una categoria)

Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    Xr, yr, test_size=0.2, random_state=42
)
print("Caratteristiche disponibili:", list(Xr.columns))
print("Esempio di valori da predire:", yr_train.values[:5])

In [ ]:
from sklearn.linear_model import LinearRegression

regresor = LinearRegression()         # 1. creare
regresor.fit(Xr_train, yr_train)      # 2. addestrare
pred_reg = regresor.predict(Xr_test)  # 3. predire

# Confrontiamo predizione vs. realtà nei primi casi
for real, pred in zip(yr_test.values[:5], pred_reg[:5]):
    print(f"Reale: {real:6.1f}   |   Predetto: {pred:6.1f}")

### Metriche di regressione

In regressione non parliamo di "risposte corrette" ma di **quanto ci discostiamo**. Due metriche comuni:
- **MAE** (errore assoluto medio): in media, di quante unità sbagliamo? Più è **basso**, meglio è
- **R²** (coefficiente di determinazione): da 0 a 1, quale parte della variazione spiega il modello. Più è **vicino a 1**, meglio è

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

mae = mean_absolute_error(yr_test, pred_reg)
r2 = r2_score(yr_test, pred_reg)
print(f"MAE (errore medio): {mae:.1f} unità")
print(f"R²: {r2:.3f}")

## 5. Visualizzare con matplotlib

Un grafico vale più di mille metriche. In regressione, un trucco molto utile è confrontare **valori reali vs. predetti**: se il modello fosse perfetto, tutti i punti cadrebbero sulla diagonale.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 6))
plt.scatter(yr_test, pred_reg, alpha=0.6, color="#1f77b4")
# Linea diagonale di riferimento (predizione perfetta)
lim = [yr_test.min(), yr_test.max()]
plt.plot(lim, lim, "--", color="red", label="predizione perfetta")
plt.xlabel("Valore reale")
plt.ylabel("Valore predetto")
plt.title("Regressione: reale vs. predetto")
plt.legend()
plt.show()

### 🟢 Esercizio 2

Torna alla **classificazione** di iris. Disegna un grafico a dispersione (`plt.scatter`) con la lunghezza del petalo sull'asse X e la larghezza del petalo sull'asse Y, **colorando ogni punto in base alla sua specie**. Si vedono gruppi separati?

*Suggerimento: scorri le specie con un ciclo `for` e disegna uno `scatter` per specie, come hai fatto con i dati nella Sessione 3.*

In [ ]:
# TODO: grafico a dispersione di iris colorato per specie
# (recupera il DataFrame df con la colonna 'especie' della Sessione 3 se ti serve)


**Soluzione:**

In [ ]:
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris

iris = load_iris(as_frame=True)
df = iris.frame.copy()
df["especie"] = df["target"].map(dict(enumerate(iris.target_names)))

plt.figure(figsize=(7, 5))
for especie in df["especie"].unique():
    sub = df[df["especie"] == especie]
    plt.scatter(sub["petal length (cm)"], sub["petal width (cm)"], label=especie, alpha=0.7)
plt.xlabel("Lunghezza petalo (cm)")
plt.ylabel("Larghezza petalo (cm)")
plt.title("Iris: le specie formano gruppi separati")
plt.legend()
plt.show()

## 6. Riepilogo della sessione e chiusura del Modulo 2

- Ogni modello di scikit-learn segue lo schema **creare → `fit` → `predict`**
- La **classificazione** predice categorie (specie); la **regressione** predice numeri (un indicatore)
- Valutiamo sui dati di **test**: con *accuratezza* e *matrice di confusione* in classificazione; con *MAE* e *R²* in regressione
- **matplotlib** ti aiuta a vedere i dati e a giudicare il modello a colpo d'occhio

Ora sai scrivere Python, gestire dati e addestrare modelli classici. **Questa è una base solida di IA applicata.**

**Per la prossima sessione (Modulo 3):** facciamo il salto verso i **grandi modelli di linguaggio** (come Claude o GPT): cosa sono, come funzionano internamente e come ci si rivolge loro bene (*prompt engineering*).

👉 **Pensa a un'attività testuale** che vorresti automatizzare (riassumere, classificare, scrivere...). La useremo come esempio lavorando con i modelli di linguaggio.